In [0]:
dbutils.widgets.text('storage_account', '', '')
dbutils.widgets.text('key_vault_scope', '', '')

In [0]:
key_vault_scope = dbutils.widgets.get('key_vault_scope') # pylint: disable=unused-variable

### Run shared notebook

In [0]:
%run /dxcore/Utilities/MailAlerts $key_vault_scope=key_vault_scope

In [0]:
storage_account = dbutils.widgets.get('storage_account')

In [0]:
%run /dxcore/Utilities/Utilities $storage_account=storage_account

### Define input parameters and load configuration

In [0]:
dbutils.widgets.text('ingestion_ts', '', '')
dbutils.widgets.text('silver_partition', '', '')
dbutils.widgets.text('params', '', '')

silver_partition  = dbutils.widgets.get('silver_partition')
json_params        = dbutils.widgets.get('params')
#
# Parse JSON from input parameter
#
print(f"JSON params: {json_params}\n")

try:
  parsed_params = json.loads(json_params)
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(f"Unable to parse input parameters, Error: {str(e)}"))
  
storage_account_uri = f"{storage_account}.dfs.core.windows.net" # pylint: disable=unused-variable
  
#
# Get ingestion timestamp from a parameter passed into this notebook from ADF. If, for some reason, it has not been
# passed in successfully, set it to the current time.
#
table_name          = parsed_params['DataMovementShortName']
silver_directory = '/'.join(parsed_params['SinkLandingDirectory'].split('/')[1:])

try:
  ingestion_ts = dbutils.widgets.get('ingestion_ts')
  if not ingestion_ts:
    ingestion_ts = str(datetime.datetime.now())
  
  dataset = c.get_dataset_by_name(table_name)
  if not dataset:
    dataset = c.get_dataset_by_table_name(table_name)
  if not dataset:
    raise Exception(f"Unable to find configuration for dataset '{dataset_name}'")
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(f'Error getting configuration, Error: {str(e)}'))
  

if not dataset:
  print(f'Dataset Name {dataset_name} is not found in the configuration file')
  dbutils.notebook.exit(get_error_payload(f'Dataset Name {dataset_name} is not found in the configuration file', ''))
  
print(f"Dataset {dataset.system}/{dataset.name} for table {dataset.table_name}")
print(f"silver_partition = {silver_partition}")
print(f"silver_directory = {silver_directory}")
print(f"ingestion_ts = {ingestion_ts}")

###Get Bronze zone and silver zone config data

In [0]:
silver_conf          = dataset.zones['silver']
silver_format        = silver_conf.databricks.format
silver_path          = dataset.dataset_uri('silver',dataset_name=silver_directory)
silver_write_mode    = dataset.mode

print(f"Silver format:\t\t\t{silver_format}")
print(f"Silver path:\t\t\t{silver_path}")

### Set the number of partitions Spark uses for shuffling to match the number of cores in the cluster

In [0]:
spark.conf.set('spark.sql.shuffle.partitions', sc.defaultParallelism)

###Some dates from the source system have unusual years (e.g. 0 or 1000). These dates can cause problems for Spark 3.0. To leave them unchanged, set settings to support legacy date format.

In [0]:
spark.conf.set('spark.sql.legacy.parquet.datetimeRebaseModeInRead', 'LEGACY')
spark.conf.set('spark.sql.legacy.parquet.datetimeRebaseModeInWrite', 'LEGACY')

### Enable caching and adaptive execution

In [0]:
spark.conf.set('spark.databricks.io.cache.enabled', 'true')
spark.conf.set('spark.sql.adaptive.enabled', 'true')

###Allow SQL MERGE to automatically update the schema in the Silver zone, if necessary

In [0]:
spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')

###Write the data in parquet format with custom names in silver###

In [0]:
try:  # pylint: disable=R1702
  gold_path = "abfss://commercial-dx@"+storage_account+".dfs.core.windows.net/gold/parquet/"+silver_directory+"/"

  ingestion_dt          = silver_partition.split('/')[0].split('=')[1]
  job_id                = silver_partition.split('/')[1].split('=')[1]

  silver_df =  spark.read.format(silver_format).load(silver_path)
  silver_df = silver_df.dropDuplicates()

  silver_df = (silver_df.filter(F.col('ingestion_dt') == ingestion_dt).filter(F.col('job_id') == job_id))
  print(f"After filtering, {silver_df.count():,} rows were loaded from silver")

  if silver_path.__contains__("safegraph"):

    if silver_path.__contains__("weekly"):
      no_of_lst_digits = 8
    elif silver_path.__contains__("monthly"):
      no_of_lst_digits = 6

    if silver_path.__contains__("core_poi-patterns"):
      silver_df = silver_df.withColumn('filepath_custom', regexp_replace('filepath','core_poi-patterns-part\d',''))  # pylint: disable=W1401
      silver_df = silver_df.withColumn('CustomName', regexp_replace('filepath_custom','/\d',''))  # pylint: disable=W1401
      silver_df = silver_df.withColumn('CustomName', regexp_replace('CustomName','\D','')).drop('filepath_custom')  # pylint: disable=W1401
      silver_df = silver_df.withColumn('CustomName', when(length(col("CustomName"))>10,col('CustomName').substr(-no_of_lst_digits, no_of_lst_digits)).otherwise(col('CustomName')))
      silver_df.createOrReplaceTempView('silver_df')

    else:
      silver_df = silver_df.withColumn('CustomName', regexp_replace('filepath','\D',''))  # pylint: disable=W1401
      silver_df = silver_df.withColumn('CustomName', when(length(col("CustomName"))>10,col('CustomName').substr(-no_of_lst_digits, no_of_lst_digits)).otherwise(col('CustomName')))
      silver_df.createOrReplaceTempView('silver_df')

  elif silver_path.__contains__("mapbox"):
    silver_df = silver_df.withColumn('CustomName', regexp_replace('agg_day_period','\D',''))  # pylint: disable=W1401
    silver_df = silver_df.withColumn('CustomName', when(length(col("CustomName"))>10,col('CustomName').substr(-6, 6)).otherwise(col('CustomName')))
    silver_df.createOrReplaceTempView('silver_df')

  elif silver_path.__contains__("pdi") or silver_path.__contains__("tdlinx"):

    silver_df.write.mode(silver_write_mode).parquet(gold_path)

  if silver_path.__contains__("mapbox") or silver_path.__contains__("safegraph"):
    if silver_path.__contains__("brand_info"):
      df_all_names = spark.sql("""SELECT Max(CustomName) as CustomName FROM silver_df""")

      dbutils.fs.rm(gold_path,True)
    else:
      df_all_names = spark.sql("""SELECT DISTINCT CustomName FROM silver_df where CustomName is not NULL""")

    l_all_names =  list(df_all_names.select("CustomName").rdd.flatMap(lambda x: x).collect())

    for name in l_all_names:
      df_custom_name = spark.sql("""SELECT * FROM silver_df WHERE CustomName = {0}""".format(name))
      df_custom_name = df_custom_name.drop('CustomName')
    #   out_file_name = file_name+'_'+name+'.snappy.parquet'  
      df_custom_name = df_custom_name.cache()

      df_custom_name.write.mode(silver_write_mode).parquet(gold_path+name)
except Exception as e:
  dbutils.notebook.exit(get_error_payload(f'Error in writing the data in {gold_path}, Error: {str(e)}'))


In [0]:
try:
  #
  # Count the number of rows in the bronze dataset
  #
  row_count = silver_df.count()
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(
    f"Unable to filter dataset {dataset.name} ingested since {ingestion_ts}, Error: {str(e)}", dataset.name))
  
    
print(f"Loaded {row_count:,} rows from silver")

###Returns success message to caller (in this case, Azure Data Factory)

In [0]:
dbutils.notebook.exit(json.dumps({
  "status": PipelineStatus.SUCCESS.value
  ,"dataset": dataset.name
  ,"row_count": row_count
  ,"partition_count": 1
  ,"is_translatable": dataset.is_translatable
  ,"error_msg": None
}))